# What is the fraud base rate, and does it drift across the time axis?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [19]:
import polars as pl
from IPython.display import Markdown, display

t = pl.scan_csv("../../kaggle/raw/train_transaction.csv")
ctx = pl.SQLContext()
ctx.register("t", t)


<SQLContext [tables:1] at 0x1492e2550>

In [20]:
import polars as pl

# Use Polars lazy scanning
t = pl.scan_csv("../../kaggle/raw/train_transaction.csv")
i = pl.scan_csv("../../kaggle/raw/train_identity.csv")

ctx = pl.SQLContext()
ctx.register("t", t)
ctx.register("i", i)

query = """
SELECT 
    COUNT(*) as Transactions,
    SUM(isFraud) as Fraud_cases,
    SUM(isFraud) / COUNT(*) as Fraud_rate,
    CAST((MAX(TransactionDT) - MIN(TransactionDT)) / 86400 AS INT) as Time_span_days,
    COUNT(id_01) as Identity_coverage
FROM t
LEFT JOIN i
ON t.TransactionID = i.TransactionID
"""
df_summary = ctx.execute(query).collect()
display(Markdown(df_summary.to_pandas().to_markdown(index=False)))

|   Transactions |   Fraud_cases |   Fraud_rate |   Time_span_days |   Identity_coverage |
|---------------:|--------------:|-------------:|-----------------:|--------------------:|
|         590540 |         20663 |      0.03499 |              181 |              144233 |

## Base Rate & Drift

3.499% overall — but the rate is not constant across the 182 days. In 30-day buckets
(`TransactionDT` is seconds from an unstated origin, so calendar months are not
recoverable):

A random split would be measuring the wrong thing. Train and test have to be separated on the time axis or the model is scored on a period whose base rate it already saw. See [point-in-time.md](point-in-time.md) §4.

---

In [21]:
query_drift = """
SELECT 
    CAST((TransactionDT - 86400) / 2592000 AS INTEGER) as Bucket,
    COUNT(*) as Transactions,
    SUM(isFraud) / COUNT(*) as Fraud_rate,
    AVG(TransactionAmt) as Mean_amount
FROM t
GROUP BY Bucket
ORDER BY Bucket
"""
df_drift = ctx.execute(query_drift).collect()

# Format df_drift to match markdown table
df_drift_md = df_drift.with_columns([
    pl.col("Bucket").map_elements(lambda x: f"{x} (partial)" if x == 6 else str(x), return_dtype=pl.String),
    pl.col("Fraud_rate").map_elements(lambda x: f"**{x*100:.2f}%**" if x < 0.03 or x > 0.043 else f"{x*100:.2f}%", return_dtype=pl.String),
    pl.col("Mean_amount").map_elements(lambda x: f"{x:.2f}", return_dtype=pl.String)
])
display(Markdown(df_drift_md.to_pandas().to_markdown(index=False)))


| Bucket      |   Transactions | Fraud_rate   |   Mean_amount |
|:------------|---------------:|:-------------|--------------:|
| 0           |         134339 | **2.53%**    |        128.28 |
| 1           |          89399 | 4.00%        |        133.53 |
| 2           |          92189 | 4.04%        |        140.03 |
| 3           |          98615 | 3.95%        |        139.53 |
| 4           |          83571 | 3.41%        |        133.7  |
| 5           |          86934 | 3.42%        |        136.1  |
| 6 (partial) |           5493 | **4.39%**    |        162.89 |